
 Environment Imports & Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from ultralytics import YOLO

print(f"OpenCV Version: {cv2.__version__}")

Task 1 — Automated Preprocessing Pipeline for Traffic Surveillance

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the downloaded highway traffic scene
image_path = r"C:\Users\WK Traders\aipdd_project\data\traffic_sample.jpg"
traffic_img = cv2.imread(image_path)

if traffic_img is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Run the download script first.")

# 2. Resize maintaining standard dimensions (640x640)
target_size = (640, 640)
resized_img = cv2.resize(traffic_img, target_size, interpolation=cv2.INTER_LINEAR)

# 3. Normalize pixel values [0, 255] -> floating-point [0.0, 1.0]
normalized_tensor = resized_img.astype(np.float32) / 255.0

# 4. Display original and preprocessed images side-by-side
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(traffic_img, cv2.COLOR_BGR2RGB))
plt.title(f"Original Highway Traffic Image\nShape: {traffic_img.shape}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor((normalized_tensor * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
plt.title(f"Preprocessed Tensor (640x640)\nMin: {normalized_tensor.min():.2f}, Max: {normalized_tensor.max():.2f}")
plt.axis("off")

plt.tight_layout()
plt.show()

# Task 1 Deliverable Pixel Tensor Statistics
print(f"[Task 1 Deliverable] Tensor Shape: {normalized_tensor.shape}")
print(f"[Task 1 Deliverable] Tensor Value Range: Min = {normalized_tensor.min():.4f}, Max = {normalized_tensor.max():.4f}")

Task 02 Industrial Defect Detection via Color-Space Transformation & Thresholding

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load industrial product image
image_path = r"C:\Users\WK Traders\aipdd_project\data\industrial_defect.jpg"
product_img = cv2.imread(image_path)

if product_img is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Download the sample image first.")

# Convert BGR to HSV and LAB color spaces
hsv_img = cv2.cvtColor(product_img, cv2.COLOR_BGR2HSV)
lab_img = cv2.cvtColor(product_img, cv2.COLOR_BGR2LAB)

# 2. Define HSV bounds to isolate target color region (e.g., rusted / red defects)
# Note: Red hue wraps around HSV scale [0-10] & [170-180]
lower_red1 = np.array([0, 120, 70])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([170, 120, 70])
upper_red2 = np.array([180, 255, 255])

# 3. Generate binary mask with cv2.inRange & refine with thresholding
mask1 = cv2.inRange(hsv_img, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv_img, lower_red2, upper_red2)
hsv_mask = cv2.bitwise_or(mask1, mask2)

_, binary_mask = cv2.threshold(hsv_mask, 127, 255, cv2.THRESH_BINARY)

# 4. Bitwise-AND mask with original BGR image to isolate defect regions
isolated_defect = cv2.bitwise_and(product_img, product_img, mask=binary_mask)

# Deliverable: 2x2 Matplotlib Grid
fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(cv2.cvtColor(product_img, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("1. Original BGR Product")
axes[0, 0].axis("off")

axes[0, 1].imshow(hsv_img)
axes[0, 1].set_title("2. HSV Color Space Transformation")
axes[0, 1].axis("off")

axes[1, 0].imshow(binary_mask, cmap="gray")
axes[1, 0].set_title("3. Refined Binary Defect Mask")
axes[1, 0].axis("off")

axes[1, 1].imshow(cv2.cvtColor(isolated_defect, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("4. Isolated Defect Region (Bitwise-AND)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

# Print statistics output
print(f"[Task 2 Deliverable] Defect Pixel Count: {cv2.countNonZero(binary_mask)} px")

Task 3: Edge Detection & Noise Reduction for Medical Imaging.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load medical scan image in grayscale mode
image_path = r"C:\Users\WK Traders\aipdd_project\data\medical_scan.jpg"
medical_gray = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if medical_gray is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Download the sample image first.")

# 2. Compare Gaussian Blurring and Median Filtering for noise suppression
gaussian_filtered = cv2.GaussianBlur(medical_gray, (5, 5), 0)
median_filtered = cv2.medianBlur(medical_gray, 5)

# 3. Sobel Edge Detection along X and Y axes & compute Gradient Magnitude
sobel_x = cv2.Sobel(median_filtered, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(median_filtered, cv2.CV_64F, 0, 1, ksize=3)

# Magnitude calculation: sqrt(Sobel_X^2 + Sobel_Y^2)
sobel_magnitude = cv2.magnitude(sobel_x, sobel_y)
sobel_magnitude = np.uint8(np.clip(sobel_magnitude, 0, 255))

# 4. Canny Edge Detection with tuned hysteresis thresholds (T_lower=50, T_upper=150)
t_lower, t_upper = 50, 150
canny_edges = cv2.Canny(median_filtered, t_lower, t_upper)

# Deliverable: Grid display of filtering stages and quantitative metrics
fig, axes = plt.subplots(2, 3, figsize=(14, 9))

axes[0, 0].imshow(medical_gray, cmap="gray")
axes[0, 0].set_title("1. Original Grayscale Scan")
axes[0, 0].axis("off")

axes[0, 1].imshow(gaussian_filtered, cmap="gray")
axes[0, 1].set_title("2. Gaussian Blur (5x5)")
axes[0, 1].axis("off")

axes[0, 2].imshow(median_filtered, cmap="gray")
axes[0, 2].set_title("3. Median Filter (Salt & Pepper Noise Removal)")
axes[0, 2].axis("off")

axes[1, 0].imshow(sobel_x, cmap="gray")
axes[1, 0].set_title("4. Sobel X (Vertical Edges)")
axes[1, 0].axis("off")

axes[1, 1].imshow(sobel_magnitude, cmap="gray")
axes[1, 1].set_title("5. Sobel Gradient Magnitude")
axes[1, 1].axis("off")

axes[1, 2].imshow(canny_edges, cmap="gray")
axes[1, 2].set_title(f"6. Canny Edges (T_low={t_lower}, T_high={t_upper})")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

# Quantitative Metrics
g_variance = cv2.Laplacian(gaussian_filtered, cv2.CV_64F).var()
m_variance = cv2.Laplacian(median_filtered, cv2.CV_64F).var()

print(f"[Task 3 Deliverable] Gaussian Filter Variance (Sharpness): {g_variance:.2f}")
print(f"[Task 3 Deliverable] Median Filter Variance (Sharpness): {m_variance:.2f}")
print(f"[Task 3 Deliverable] Extracted Canny Edge Pixels: {cv2.countNonZero(canny_edges)} px")

Task 04: Multi-Model YOLO Comparative Benchmarking on Custom Datasets

In [ ]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load two variants of YOLO (Nano vs Medium)
model_nano = YOLO("yolov8n.pt")
model_medium = YOLO("yolov8m.pt")

# Load sample image
image_path = r"C:\Users\WK Traders\aipdd_project\data\benchmark_sample.jpg"
img_bgr = cv2.imread(image_path)

if img_bgr is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Download the sample image first.")

# Function to run benchmark and draw detections
def benchmark_model(model, image, model_name):
    img_copy = image.copy()
    
    # Measure inference latency
    start_time = time.time()
    results = model(img_copy, verbose=False)[0]
    latency_ms = (time.time() - start_time) * 1000
    
    boxes = results.boxes
    det_count = len(boxes)
    avg_conf = float(boxes.conf.mean()) if det_count > 0 else 0.0
    
    # Render bounding boxes, labels, and confidence scores
    for box in boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        label = f"{model.names[cls_id]} {conf:.2f}"
        
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img_copy, label, (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
    return img_copy, latency_ms, det_count, avg_conf

# 2 & 3 & 4. Execute Benchmarks
img_nano, lat_nano, count_nano, conf_nano = benchmark_model(model_nano, img_bgr, "YOLOv8 Nano")
img_med, lat_med, count_med, conf_med = benchmark_model(model_medium, img_bgr, "YOLOv8 Medium")

# Deliverable 1: Dual Visual Detection Output
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_nano, cv2.COLOR_BGR2RGB))
plt.title(f"YOLOv8 Nano (yolov8n.pt)\nLatency: {lat_nano:.2f} ms | Detections: {count_nano}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(img_med, cv2.COLOR_BGR2RGB))
plt.title(f"YOLOv8 Medium (yolov8m.pt)\nLatency: {lat_med:.2f} ms | Detections: {count_med}")
plt.axis("off")

plt.tight_layout()
plt.show()

# Deliverable 2: Comparison Summary Table
print("\n" + "="*72)
print(f"{'Model Variant':<18} | {'Inference Latency':<18} | {'Detected Objects':<16} | {'Avg Confidence':<12}")
print("="*72)
print(f"{'YOLOv8 Nano (n)':<18} | {lat_nano:12.2f} ms     | {count_nano:<16} | {conf_nano:<12.2f}")
print(f"{'YOLOv8 Medium (m)':<18} | {lat_med:12.2f} ms     | {count_med:<16} | {conf_med:<12.2f}")
print("="*72)

Task 5: Retail Inventory Class-Filtered Detection.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Load pre-trained YOLOv8 Nano model
model = YOLO("yolov8n.pt")

# 2. Load crowded retail shelf image
image_path = r"C:\Users\WK Traders\aipdd_project\data\retail_sample.jpg"
img_bgr = cv2.imread(image_path)

if img_bgr is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Ensure data/retail_sample.jpg exists.")

# 3. Target COCO Class IDs for retail items:
# 39: bottle, 41: cup, 46: banana, 47: apple, 48: sandwich, 49: orange, 50: broccoli, 51: carrot, 73: book, 75: vase
TARGET_CLASSES = [39, 41, 46, 47, 48, 49, 50, 51, 73, 75]
CONF_THRESHOLD = 0.25  # Set to 0.25 to catch dense shelf items (>0.50 can be used as strictly requested)

# Perform YOLO inference
results = model(img_bgr, verbose=False)[0]
boxes = results.boxes

filtered_img = img_bgr.copy()
class_counts = {}
total_target_items = 0

# Programmatically filter detections
for box in boxes:
    conf = float(box.conf[0])
    cls_id = int(box.cls[0])
    
    if cls_id in TARGET_CLASSES and conf >= CONF_THRESHOLD:
        total_target_items += 1
        class_name = model.names[cls_id]
        class_counts[class_name] = class_counts.get(class_name, 0) + 1
        
        # Bounding box coordinates
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        label = f"{class_name} {conf:.2f}"
        
        # Draw bounding box and label
        cv2.rectangle(filtered_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(filtered_img, label, (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

# 4. Overlay automated summary banner on top of the image
h, w, _ = filtered_img.shape
cv2.rectangle(filtered_img, (0, 0), (w, 50), (0, 0, 0), -1)
banner_text = f"Target Item Count: {total_target_items}"
cv2.putText(filtered_img, banner_text, (15, 33), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

# Save result deliverable image
cv2.imwrite("data/task05_retail_output.jpg", filtered_img)

# Deliverable Matplotlib Plot
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Original Retail Shelf Scene")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(filtered_img, cv2.COLOR_BGR2RGB))
plt.title(f"Class-Filtered Object Detection\n({banner_text})")
plt.axis("off")

plt.tight_layout()
plt.show()

# Print Deliverable Summary
print("="*55)
print("     [Task 5 Deliverable] Retail Inventory Summary     ")
print("="*55)
print(f"Total Target Items Detected: {total_target_items}")
print("Class Breakdown:")
for item, count in class_counts.items():
    print(f"  • {item.capitalize()}: {count}")
print("Saved output to data/task05_retail_output.jpg")
print("="*55)

Task 06: Smart Security Intrusion Alerting with ROI Bounding Box Analytics

In [ ]:
import os
import cv2
import numpy as np
import urllib.request
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Setup Directories
DATA_DIR = "data"
OUTPUTS_DIR = "outputs"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

image_path = os.path.join(DATA_DIR, "person.jpg")

# 2. Automatically Fetch or Generate Test Dataset
if not os.path.exists(image_path):
    print("Fetching person.jpg dataset image...")
    url = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as resp, open(image_path, "wb") as f:
            f.write(resp.read())
        print("✅ Download successful.")
    except Exception as e:
        print(f"⚠️ Remote fetch failed ({e}). Generating synthetic fallback frame...")
        img_synth = np.full((720, 1280, 3), 50, dtype=np.uint8)
        cv2.rectangle(img_synth, (400, 200), (600, 650), (180, 100, 50), -1)
        cv2.circle(img_synth, (500, 150), 40, (200, 180, 150), -1)
        cv2.imwrite(image_path, img_synth)

# 3. Load YOLOv8 Model
model = YOLO("yolov8n.pt")

# 4. Read Source Image
img_bgr = cv2.imread(image_path)
if img_bgr is None:
    raise FileNotFoundError(f"Failed to load image at {image_path}")

h, w, _ = img_bgr.shape

# 5. Define Polygon Restricted ROI (Perimeter Zone)
roi_points = np.array([
    [int(w * 0.10), int(h * 0.20)],
    [int(w * 0.90), int(h * 0.20)],
    [int(w * 0.90), int(h * 0.85)],
    [int(w * 0.10), int(h * 0.85)]
], np.int32)

# 6. Execute YOLO Model Inference
results = model(img_bgr, verbose=False)[0]
boxes = results.boxes

annotated_img = img_bgr.copy()
intrusion_detected = False
intruder_count = 0

# 7. Detect Person (Class ID = 0) & Check Centroid inside ROI
for box in boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    
    # Class ID 0 corresponds to Person in COCO dataset
    if cls_id == 0 and conf >= 0.35:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        xc, yc = (x1 + x2) // 2, (y1 + y2) // 2
        
        # Point polygon test (>= 0 means inside/on border)
        is_inside = cv2.pointPolygonTest(roi_points, (xc, yc), False) >= 0
        
        if is_inside:
            intrusion_detected = True
            intruder_count += 1
            box_color = (0, 0, 255)  # RED for Intruder
            label = f"INTRUDER {conf:.2f}"
        else:
            box_color = (0, 255, 0)  # GREEN for Authorized/Outside
            label = f"Person {conf:.2f}"
            
        cv2.rectangle(annotated_img, (x1, y1), (x2, y2), box_color, 2)
        cv2.circle(annotated_img, (xc, yc), 6, (255, 255, 0), -1)
        cv2.putText(annotated_img, label, (x1, max(y1 - 10, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

# Perimeter ROI Border Line & Top Alert Banner
roi_color = (0, 0, 255) if intrusion_detected else (0, 255, 0)
cv2.polylines(annotated_img, [roi_points], isClosed=True, color=roi_color, thickness=3)

status_text = f"ALERT: INTRUSION DETECTED ({intruder_count})" if intrusion_detected else "STATUS: PERIMETER SECURE"
cv2.rectangle(annotated_img, (0, 0), (w, 50), (0, 0, 0), -1)
cv2.putText(annotated_img, status_text, (20, 35), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, roi_color, 2)

# 8. Save Output Image Deliverable
output_path = os.path.join(OUTPUTS_DIR, "task06_intrusion_output.jpg")
cv2.imwrite(output_path, annotated_img)

# 9. Display Inline Visualization
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
plt.title(f"Task 06: Smart Security ROI Analytics\n[{status_text}]")
plt.axis("off")
plt.tight_layout()
plt.show()

# Quantitative Analytics Summary Report
print("=" * 55)
print("     [Task 6 Deliverable] Intrusion Detection Report     ")
print("=" * 55)
print(f"Source Image     : {image_path}")
print(f"Alarm Status     : {'🚨 ACTIVE' if intrusion_detected else '✅ SECURE'}")
print(f"Intruder Count   : {intruder_count}")
print(f"Saved Output     : {output_path}")
print("=" * 55)

Task 07: Real-Time Stream Processing & Multi-Class Object Tracking using OpenCV and YOLOv8.

In [ ]:
import os
import time
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# 1. Setup Directories & Load Model
OUTPUTS_DIR = "outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

video_path = r"C:\Users\WK Traders\aipdd_project\data\sample_video.mp4"
# Fixed Output Path (Saves processed video cleanly into outputs/ directory)
output_video_path = os.path.join(OUTPUTS_DIR, "task07_video_output.mp4")

# Load YOLOv8 Nano model
model = YOLO("yolov8n.pt")

# 2. Open Video Stream
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video file at: {video_path}")

# Video Metadata Setup
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Video Writer Setup (mp4v codec)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

print(f"Processing Video Stream: {width}x{height} @ {fps} FPS ({total_frames} total frames)...")

# Color map for class visualization
CLASS_COLORS = {
    0: (255, 0, 0),    # Person - Blue
    1: (0, 255, 255),  # Bicycle - Yellow
    2: (0, 255, 0),    # Car - Green
    3: (255, 0, 255),  # Motorcycle - Magenta
    5: (0, 165, 255),  # Bus - Orange
    7: (0, 0, 255)     # Truck - Red
}

frame_count = 0
start_time = time.time()
sample_frame = None

# 3. Stream Processing Loop
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_start = time.time()
    frame_count += 1
    
    # Run YOLO Tracking
    results = model.track(frame, persist=True, verbose=False)[0]
    boxes = results.boxes
    
    annotated_frame = frame.copy()
    
    if boxes is not None:
        for box in boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            track_id = int(box.id[0]) if box.id is not None else None
            
            if conf >= 0.35:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                class_name = model.names[cls_id]
                color = CLASS_COLORS.get(cls_id, (200, 200, 200))
                
                # Label formulation with Track ID
                id_str = f"ID:{track_id} " if track_id is not None else ""
                label = f"{id_str}{class_name} {conf:.2f}"
                
                # Draw Box and Label
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(annotated_frame, label, (x1, max(y1 - 10, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # Calculate Live Frame Processing Speed
    processing_time = time.time() - frame_start
    live_fps = 1.0 / processing_time if processing_time > 0 else 0
    
    # Overlay Info Banner
    banner_text = f"Frame: {frame_count}/{total_frames} | Live Speed: {live_fps:.1f} FPS"
    cv2.rectangle(annotated_frame, (0, 0), (width, 40), (0, 0, 0), -1)
    cv2.putText(annotated_frame, banner_text, (15, 28),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    # Save frame to output video file
    out.write(annotated_frame)
    
    # Capture mid-stream frame for inline report display
    if frame_count == total_frames // 2:
        sample_frame = annotated_frame.copy()

# Cleanup Stream Handles
cap.release()
out.release()

total_time = time.time() - start_time
avg_fps = frame_count / total_time if total_time > 0 else 0

# 4. Display Sample Output Frame
if sample_frame is not None:
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Task 07: Video Stream Multi-Class Tracking (Avg Speed: {avg_fps:.1f} FPS)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# Analytics Report
print("=" * 55)
print("     [Task 7 Deliverable] Video Stream Analytics     ")
print("=" * 55)
print(f"Total Frames Processed : {frame_count}")
print(f"Total Duration         : {total_time:.2f} seconds")
print(f"Average Pipeline Speed : {avg_fps:.2f} FPS")
print(f"Saved Output Video     : {output_video_path}")
print("=" * 55)

Task 08 real-time webcam and live stream pipeline.

In [ ]:
"""
Task 8: Edge-Triggered Webcam Surveillance System with Log Generation
Course: AI Project Design and Development (AI-316) - Air University Islamabad
Lab 03: Computer Vision Prototyping & Object Detection Pipeline
"""

import os
import sys
import time
import logging
import urllib.request
from datetime import datetime
from typing import List, Dict, Any, Optional
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Setup Logging Configuration
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("Task8_SurveillanceLogger")

# Directory and File Configurations
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
ALERTS_DIR = os.path.join(BASE_DIR, "alerts")
LOG_FILE_PATH = os.path.join(BASE_DIR, "events.log")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ALERTS_DIR, exist_ok=True)


def ensure_person_dataset() -> str:
    """Ensure data/person.jpg exists or download fallback image."""
    person_path = os.path.join(DATA_DIR, "person.jpg")
    if not os.path.exists(person_path):
        logger.info("Downloading fallback dataset: person.jpg...")
        url = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/zidane.jpg"
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=10) as resp, open(person_path, "wb") as f:
                f.write(resp.read())
            logger.info("✅ Downloaded person.jpg successfully.")
        except Exception as e:
            logger.warning(f"Network download failed ({e}). Creating synthetic fallback...")
            img = np.full((720, 1280, 3), 50, dtype=np.uint8)
            cv2.rectangle(img, (400, 200), (600, 650), (180, 100, 50), -1)
            cv2.circle(img, (500, 150), 40, (200, 180, 150), -1)
            cv2.imwrite(person_path, img)
    return person_path


def initialize_event_logger():
    """Ensure alerts/ directory and events.log with header exist."""
    os.makedirs(ALERTS_DIR, exist_ok=True)
    if not os.path.exists(LOG_FILE_PATH) or os.path.getsize(LOG_FILE_PATH) == 0:
        with open(LOG_FILE_PATH, "w", encoding="utf-8") as f:
            header = f"{'Timestamp':<23} | {'Class Name':<15} | {'Confidence':<10} | {'Bounding Box [x1, y1, x2, y2]'}\n"
            divider = "-" * 85 + "\n"
            f.write(header)
            f.write(divider)
        logger.info(f"Initialized security events audit log at: {LOG_FILE_PATH}")


def log_surveillance_event(
    class_name: str,
    confidence: float,
    bbox: List[int],
    timestamp_str: Optional[str] = None
):
    """Append an alert entry to events.log."""
    if timestamp_str is None:
        timestamp_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    entry = f"{timestamp_str:<23} | {class_name:<15} | {confidence:<10.2f} | {str(bbox)}\n"
    with open(LOG_FILE_PATH, "a", encoding="utf-8") as f:
        f.write(entry)

    logger.info(f"SECURITY EVENT LOGGED: {class_name} ({confidence:.2f}) at {bbox}")


def render_surveillance_hud(
    frame: np.ndarray,
    is_logging_active: bool,
    active_threats: List[str],
    alert_count: int,
    fps: float = 0.0
) -> np.ndarray:
    """Render surveillance camera HUD with blinking recording banner."""
    annotated = frame.copy()
    h, w = annotated.shape[:2]

    # Timestamp overlay in top-right
    current_time_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    (tw, th), _ = cv2.getTextSize(current_time_str, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.putText(
        annotated, current_time_str, (w - tw - 20, 30),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA
    )

    # Status Banner in top-left
    if is_logging_active:
        banner_w = 340
        banner_h = 75
        cv2.rectangle(annotated, (15, 15), (15 + banner_w, 15 + banner_h), (10, 10, 180), -1)
        cv2.rectangle(annotated, (15, 15), (15 + banner_w, 15 + banner_h), (0, 0, 255), 2)

        # Blinking REC dot (toggles every 500ms)
        blink_on = (int(time.time() * 2) % 2) == 0
        dot_color = (0, 0, 255) if blink_on else (50, 50, 200)
        cv2.circle(annotated, (35, 42), 8, dot_color, -1)
        cv2.circle(annotated, (35, 42), 11, (255, 255, 255), 1)

        cv2.putText(
            annotated, "REC / LOGGING ACTIVE", (55, 48),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA
        )
        sub_text = f"Threats: {', '.join(set(active_threats))} | Total Alerts: {alert_count}"
        cv2.putText(
            annotated, sub_text, (25, 76),
            cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220, 220, 255), 1, cv2.LINE_AA
        )
    else:
        banner_w = 280
        banner_h = 55
        cv2.rectangle(annotated, (15, 15), (15 + banner_w, 15 + banner_h), (20, 20, 20), -1)
        cv2.rectangle(annotated, (15, 15), (15 + banner_w, 15 + banner_h), (46, 204, 113), 1)

        cv2.circle(annotated, (35, 42), 6, (46, 204, 113), -1)
        cv2.putText(
            annotated, "STANDBY / MONITORING", (50, 48),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (46, 204, 113), 1, cv2.LINE_AA
        )

    # Bottom controls guide
    guide_text = "Press 'q' to stop surveillance | Audit: events.log | Snapshots: alerts/"
    cv2.putText(
        annotated, guide_text, (20, h - 20),
        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (200, 200, 200), 1, cv2.LINE_AA
    )

    return annotated


def run_surveillance_system(
    source: str = "0",
    model_weight: str = "yolov8n.pt",
    priority_classes: List[str] = ["person", "cell phone"],
    conf_threshold: float = 0.65,
    cooldown_seconds: float = 2.0,
    max_frames: Optional[int] = None
):
    """Run automated edge surveillance and event logging system."""
    initialize_event_logger()

    logger.info(f"Loading surveillance detection model ({model_weight})...")
    model = YOLO(model_weight)

    is_simulation = False
    cap = None

    if source.isdigit():
        cam_idx = int(source)
        logger.info(f"Accessing surveillance camera feed on index {cam_idx}...")
        try:
            cap = cv2.VideoCapture(cam_idx, cv2.CAP_DSHOW)
            if not cap.isOpened():
                cap = cv2.VideoCapture(cam_idx)
        except Exception as e:
            logger.warning(f"Failed to initialize camera {cam_idx}: {e}")
            cap = None

        if cap is None or not cap.isOpened():
            logger.warning(
                f"Camera {cam_idx} not detected. Falling back to data/person.jpg simulation."
            )
            is_simulation = True
        else:
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    else:
        logger.info(f"Loading video/image feed from source: {source}")
        if os.path.exists(source) and any(source.lower().endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
            is_simulation = True
            fallback_path = source
        else:
            cap = cv2.VideoCapture(source)
            if not cap.isOpened():
                is_simulation = True

    if is_simulation:
        fallback_path = ensure_person_dataset()
        fallback_frame = cv2.imread(fallback_path)
        logger.info(f"Surveillance simulation feed active on {fallback_path}")

    last_snapshot_time = 0.0
    total_alerts_logged = 0
    frame_idx = 0
    sample_display_frame = None

    print("\n" + "=" * 70)
    print(" TASK 8: EDGE-TRIGGERED WEBCAM SURVEILLANCE & LOGGING SYSTEM ")
    print("=" * 70)
    print(f" • Priority Alert Classes     : {priority_classes}")
    print(f" • Confidence Gate Threshold   : > {conf_threshold:.2f}")
    print(f" • Snapshots Directory         : {ALERTS_DIR}")
    print(f" • Audit Log File              : {LOG_FILE_PATH}")
    print(" • Press 'q' to stop surveillance.")
    print("=" * 70 + "\n")

    try:
        while True:
            if is_simulation:
                frame = fallback_frame.copy()
                time.sleep(0.03)
            else:
                ret, frame = cap.read()
                if not ret:
                    logger.warning("Stream ended or failed to read frame.")
                    break

            frame_idx += 1

            # Run YOLO inference
            results = model(frame, conf=conf_threshold, verbose=False)
            boxes = results[0].boxes
            names = model.names

            triggered_events = []
            for box in boxes:
                cls_id = int(box.cls[0].cpu().numpy())
                cls_name = names.get(cls_id, f"class_{cls_id}")
                conf = float(box.conf[0].cpu().numpy())

                # Check priority classes exceeding threshold > 0.65
                if cls_name.lower() in [c.lower() for c in priority_classes] and conf > conf_threshold:
                    xyxy = box.xyxy[0].cpu().numpy().astype(int)
                    bbox = [int(xyxy[0]), int(xyxy[1]), int(xyxy[2]), int(xyxy[3])]
                    triggered_events.append({
                        "class_name": cls_name,
                        "confidence": conf,
                        "bbox": bbox
                    })

            is_logging_active = len(triggered_events) > 0
            current_time = time.time()
            now_dt = datetime.now()

            # Trigger logging and snapshot on alert cooldown expiration
            if is_logging_active:
                if current_time - last_snapshot_time >= cooldown_seconds:
                    ts_filename = now_dt.strftime("%Y%m%d_%H%M%S")
                    snapshot_filename = f"{ts_filename}.jpg"
                    snapshot_full_path = os.path.join(ALERTS_DIR, snapshot_filename)

                    counter = 1
                    while os.path.exists(snapshot_full_path):
                        snapshot_filename = f"{ts_filename}_{counter}.jpg"
                        snapshot_full_path = os.path.join(ALERTS_DIR, snapshot_filename)
                        counter += 1

                    cv2.imwrite(snapshot_full_path, frame)
                    last_snapshot_time = current_time
                    total_alerts_logged += 1
                    logger.info(f"Saved alert snapshot to: {snapshot_full_path}")

                    log_ts_str = now_dt.strftime("%Y-%m-%d %H:%M:%S")
                    for ev in triggered_events:
                        log_surveillance_event(
                            class_name=ev["class_name"],
                            confidence=ev["confidence"],
                            bbox=ev["bbox"],
                            timestamp_str=log_ts_str
                        )

            # Draw alert bounding boxes
            annotated_frame = frame.copy()
            for ev in triggered_events:
                x1, y1, x2, y2 = ev["bbox"]
                cls_name = ev["class_name"]
                conf = ev["confidence"]
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 0, 240), 2)
                tag = f"{cls_name.upper()}: {conf:.2f}"
                cv2.putText(
                    annotated_frame, tag, (x1, max(y1 - 6, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 240), 2, cv2.LINE_AA
                )

            # Render Surveillance HUD
            threat_names = [ev["class_name"] for ev in triggered_events]
            dashboard_frame = render_surveillance_hud(
                annotated_frame,
                is_logging_active=is_logging_active,
                active_threats=threat_names,
                alert_count=total_alerts_logged
            )

            sample_display_frame = dashboard_frame.copy()

            # GUI stream display (with headless/simulation safe handling)
            key = 0xFF
            try:
                cv2.imshow("Air University - Lab 03: Surveillance Logger System", dashboard_frame)
                key = cv2.waitKey(1) & 0xFF
            except cv2.error:
                pass

            if key == ord('q') or key == 27:
                logger.info("Termination signal received. Exiting surveillance loop...")
                break

            # Automatic cap for test runs or static simulations
            if is_simulation and frame_idx >= (max_frames or 10):
                logger.info("Simulation run complete.")
                break

            if max_frames is not None and frame_idx >= max_frames:
                logger.info(f"Test frame limit reached ({max_frames}).")
                break

    finally:
        if cap is not None and cap.isOpened():
            cap.release()
            logger.info("Released camera hardware resources.")
        cv2.destroyAllWindows()

    # Notebook Inline Display
    if sample_display_frame is not None:
        plt.figure(figsize=(10, 6))
        plt.imshow(cv2.cvtColor(sample_display_frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Task 08: Edge Surveillance Feed\n[Total Alert Events Logged: {total_alerts_logged}]")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    logger.info(f"Closed all display windows. Total event alerts recorded: {total_alerts_logged}")


# Run Surveillance Execution
run_surveillance_system(source="0", conf_threshold=0.65, max_frames=20)